# Lab 06: Sessions & Persistent Memory

**Goal:** make conversations survive restarts with a session manager, and keep non-conversation data in `agent.state`.

Course module: `samples/09-persistent-memory`

An `Agent` keeps history in memory, so it's lost when the process ends. A **session manager** saves messages and state after every change and restores them when you create an agent with the same `session_id`.

| Session manager | Storage |
|---|---|
| `FileSessionManager` | local directory (dev) |
| `S3SessionManager` | S3 bucket (production, multi-instance) |
| AgentCore Memory | managed (see course module 14) |

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

In [ ]:
import shutil, pathlib
from strands import Agent
from strands.session.file_session_manager import FileSessionManager

STORAGE = pathlib.Path("workspace/sessions")
SESSION_ID = "customer-session-001"
shutil.rmtree(STORAGE / f"session_{SESSION_ID}", ignore_errors=True)   # start clean for the demo

def make_agent():
    return Agent(
        model=get_model(),
        system_prompt="You are a friendly support agent. Use prior conversation context; never re-ask for info you already have.",
        session_manager=FileSessionManager(session_id=SESSION_ID, storage_dir=str(STORAGE)),
        callback_handler=None,
    )

## 1. First conversation

In [ ]:
agent = make_agent()
print("restored messages:", len(agent.messages))

agent.state.set("customer_id", "C-1042")     # structured state, saved with the next sync
agent.state.set("open_ticket", "usb-c-hub-charging")

print(agent("Hi, I'm Jordan, customer ID C-1042. My USB-C hub stopped charging my laptop."))

## 2. "Restart": a brand-new agent object with the same session id
This is what happens after a process restart, a new Lambda invocation, or a new notebook kernel.

In [ ]:
del agent
agent = make_agent()
print("restored messages:", len(agent.messages))
print("restored state:   ", agent.state.get())
print()
print(agent("Any update? Also, remind me what my customer ID is."))

## 3. What's on disk

In [ ]:
for p in sorted(STORAGE.rglob("*")):
    if p.is_file():
        print(p.relative_to(STORAGE), f"({p.stat().st_size} bytes)")

## 4. S3 for production (reference)
Swap the manager. The agent code stays the same:

```python
from strands.session.s3_session_manager import S3SessionManager

session_manager = S3SessionManager(
    session_id=user_session_id,
    bucket="my-agent-sessions",
    prefix="prod/",
    region_name="us-east-1",
)
```

Long sessions eventually overflow the context window. The course covers this in `08-context-management`: `context_manager="auto"` offloads large tool results and summarizes old turns.

## 🧪 Exercise
Make the session id come from user input, so different "users" get isolated histories. Verify that two ids don't share memory.

**Next:** [Lab 07: Multi-Agent: Agents as Tools](07-agents-as-tools.ipynb)

In [ ]:
# your code here